# Gold — Governance Marts (files by dependents + report risk)

Builds curated **gold** tables in `lh_fabric_management` (`fabricmanagement` schema)
for the semantic model / report:

| Table | Grain | Contents |
|---|---|---|
| `report_model_map` | one row per report | report → its semantic model + workspace (from the admin reports API) |
| `gold_file_dependencies` | one row per data source | file/source with how many **models**, **reports**, **workspaces** depend on it, + `risk_tier` |
| `gold_report_risk` | one row per report | High/Med/Low source counts, `risk_score`, and the single riskiest source behind the report's model |

**Risk score** = `3×High + 2×Medium + 1×Low` distinct data sources behind the report's model.

## Inputs / prerequisites
- The **connection notebook** must have run so `connection_semantic_model_map` (model → datasource + `risk_tier`) exists.
- Run identity must be a **Fabric administrator** (`/admin/reports`, `/admin/groups`).

In [ ]:
# ── Config & auth ────────────────────────────────────────────────────────────
from datetime import datetime, timezone
import time
import requests
import notebookutils
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from delta.tables import DeltaTable

POWERBI_API      = "https://api.powerbi.com/v1.0/myorg"
LAKEHOUSE_NAME   = "lh_fabric_management"
LAKEHOUSE_SCHEMA = "fabricmanagement"          # None for a classic lakehouse
SRC_MODEL_MAP    = "connection_semantic_model_map"   # input (from the connection notebook)
TBL_REPORT_MAP   = "report_model_map"
TBL_GOLD_FILES   = "gold_file_dependencies"
TBL_GOLD_REPORTS = "gold_report_risk"

RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
print("Run timestamp (UTC):", RUN_TS.isoformat())

_TOKEN = {"v": None, "exp": 0.0}
def _headers():
    if time.time() > _TOKEN["exp"]:
        _TOKEN["v"] = notebookutils.credentials.getToken("pbi")
        _TOKEN["exp"] = time.time() + 3000
    return {"Authorization": f"Bearer {_TOKEN['v']}", "Content-Type": "application/json"}

def _get(url):
    r = requests.get(url, headers=_headers(), timeout=120)
    if r.status_code in (401, 403):
        raise PermissionError(f"HTTP {r.status_code} on {url} -> needs Fabric admin (Tenant.Read.All).")
    r.raise_for_status()
    return r.json()

def _paged(url, top=5000):
    # Page a Power BI admin list via $top/$skip.
    out, skip = [], 0
    while True:
        sep = "&" if "?" in url else "?"
        batch = _get(f"{url}{sep}$top={top}&$skip={skip}").get("value", [])
        out.extend(batch)
        if len(batch) < top:
            break
        skip += top
    return out

def _tables_path(name):
    lh = notebookutils.lakehouse.get(name)
    props = (lh.get("properties") or {}) if isinstance(lh, dict) else {}
    return (props.get("oneLakeTablesPath") or f'{props.get("abfsPath", "").rstrip("/")}/Tables')

TABLES_PATH = _tables_path(LAKEHOUSE_NAME)
def _table_uri(name):
    sub = name if not LAKEHOUSE_SCHEMA else f"{LAKEHOUSE_SCHEMA}/{name}"
    return f"{TABLES_PATH}/{sub}"

In [ ]:
# ── 1. Report -> model lineage (tenant-wide) + workspace names ───────────────
reports = _paged(f"{POWERBI_API}/admin/reports")
groups  = _paged(f"{POWERBI_API}/admin/groups")
ws_name = {g.get("id"): g.get("name") for g in groups}
print(f"reports: {len(reports)} | workspaces: {len(groups)}")

report_rows = []
for r in reports:
    report_rows.append({
        "report_id": r.get("id"),
        "report_name": r.get("name"),
        "report_type": r.get("reportType"),
        "dataset_id": r.get("datasetId"),
        "workspace_id": r.get("workspaceId"),
        "workspace_name": ws_name.get(r.get("workspaceId")),
        "created_by": r.get("createdBy"),
        "modified_date": r.get("modifiedDateTime"),
        "web_url": r.get("webUrl"),
        "scan_timestamp": RUN_TS,
    })

rmm_schema = StructType([
    StructField("report_id", StringType()),
    StructField("report_name", StringType()),
    StructField("report_type", StringType()),
    StructField("dataset_id", StringType()),
    StructField("workspace_id", StringType()),
    StructField("workspace_name", StringType()),
    StructField("created_by", StringType()),
    StructField("modified_date", StringType()),
    StructField("web_url", StringType()),
    StructField("scan_timestamp", TimestampType()),
])
_names = [f.name for f in rmm_schema.fields]
rmm = spark.createDataFrame([tuple(d.get(n) for n in _names) for d in report_rows], schema=rmm_schema)

In [ ]:
# ── 2. Read model -> datasource (latest snapshot) ────────────────────────────
if not DeltaTable.isDeltaTable(spark, _table_uri(SRC_MODEL_MAP)):
    raise RuntimeError(f"{SRC_MODEL_MAP} not found -- run the connection notebook first.")

cmm_all = spark.read.format("delta").load(_table_uri(SRC_MODEL_MAP))
latest = cmm_all.agg(F.max("snapshot_date")).first()[0]
cmm = cmm_all.where(F.col("snapshot_date") == F.lit(latest))
print(f"model<->datasource rows (snapshot {latest}): {cmm.count()}")

In [ ]:
# ── 2b. Re-classify sources from the path (+ file extension) ─────────────────
# Refines the connection-notebook classification: normalizes backslashes so bare
# Windows/UNC paths (no file:// scheme) that were "Unknown"/"Other" now match, and
# parses file_extension so any residual Unknown can be sliced by .xlsx/.csv/...
import re
from urllib.parse import unquote
from pyspark.sql.types import StructType, StructField, StringType

BS = chr(92)   # backslash (avoids escaping headaches)
DATA_EXTS = {"xlsx", "xls", "xlsb", "xlsm", "csv", "tsv", "txt", "json", "xml",
             "parquet", "pdf", "accdb", "mdb", "dat", "dbf"}

def _classify(path):
    # -> (source_kind, source_host, risk_tier, file_extension)
    if not path:
        return ("Unknown", None, "Unknown", None)
    p = unquote(path).strip()
    low = p.lower()
    norm = low.replace(BS, "/")                      # \server\share -> /server/share
    seg = norm.rstrip("/").split("/")[-1] if norm else ""
    ext = None
    if "." in seg:
        cand = seg.rsplit(".", 1)[-1]
        if cand and len(cand) <= 5 and cand.isalnum():
            ext = cand
    # Fabric / Power BI native (check before the http and ';' fallbacks)
    if "onelake.dfs.fabric.microsoft.com" in low:
        return ("FabricOneLake", "onelake", "Low", ext)
    if low.startswith("powerbi://"):
        return ("PowerBIDataset", re.sub(r"^powerbi://", "", norm).split("/")[0], "Low", ext)
    if ".datawarehouse.fabric.microsoft.com" in low or ".datawarehouse.pbidedicated.windows.net" in low:
        return ("FabricWarehouse", p.split(";")[0], "Low", ext)
    if ".database.windows.net" in low:
        return ("AzureSQL", p.split(";")[0], "Low", ext)
    # files
    m = re.match(r"(?:file:///)?([a-z]):/", norm)     # file:///K:/  or  K:/  (was K:\)
    if m:
        return ("LocalMappedDrive", m.group(1).upper() + ":", "High", ext)
    if norm.startswith("file://") or norm.startswith("//"):   # UNC share
        host = re.sub(r"^(file:)?/+", "", norm).split("/")[0]
        return ("FileShareUNC", host or None, "Medium", ext)
    if "sharepoint.com" in low:
        host = re.sub(r"^https?://", "", norm).split("/")[0]
        personal = "/personal/" in low
        return ("SharePointPersonal" if personal else "SharePoint", host,
                "High" if personal else "Medium", ext)
    if low.startswith("dsn="):
        return ("ODBC_DSN", p.split("=", 1)[1], "Medium", ext)
    if ext in DATA_EXTS:                              # data file we didn't scheme-match
        return ("File", None, "Medium", ext)
    if low.startswith("http"):
        return ("Web", re.sub(r"^https?://", "", norm).split("/")[0], "Medium", ext)
    if ";" in p:                                      # host;db -> on-prem/other SQL (gateway-bound)
        return ("SqlServer", p.split(";")[0], "Medium", ext)
    if not (set(p) & {"/", ".", ";", "=", BS}):       # bare connector/service name (Teams, PowerBIDatasets, ...)
        return ("Service", p, "Low", ext)
    return ("Other", None, "Unknown", ext)

_cls_schema = StructType([
    StructField("source_kind", StringType()),
    StructField("source_host", StringType()),
    StructField("risk_tier", StringType()),
    StructField("file_extension", StringType()),
])
_classify_udf = F.udf(_classify, _cls_schema)

cmm = (cmm.withColumn("_c", _classify_udf(F.col("path")))
          .withColumn("source_kind", F.col("_c.source_kind"))
          .withColumn("source_host", F.col("_c.source_host"))
          .withColumn("risk_tier", F.col("_c.risk_tier"))
          .withColumn("file_extension", F.col("_c.file_extension"))
          .drop("_c"))
print("Re-classified. source_kind distribution:")
cmm.groupBy("source_kind").count().orderBy(F.desc("count")).show(truncate=False)

In [ ]:
# ── 3. gold_file_dependencies : one row per data source, who depends on it ────
# reports per file, via report -> model -> file
rep_file = (rmm.alias("r")
            .join(cmm.alias("m"), F.col("r.dataset_id") == F.col("m.semantic_model_id"))
            .select("r.report_id", "m.path"))
reports_per_file = rep_file.groupBy("path").agg(F.countDistinct("report_id").alias("reports_using"))

gold_files = (cmm.groupBy("path").agg(
        F.first("source_kind", True).alias("source_kind"),
        F.first("source_host", True).alias("source_host"),
        F.first("risk_tier", True).alias("risk_tier"),
        F.first("datasource_type", True).alias("datasource_type"),
        F.first("file_extension", True).alias("file_extension"),
        F.countDistinct("semantic_model_id").alias("models_using"),
        F.countDistinct("workspace_id").alias("workspaces_using"))
    .join(reports_per_file, "path", "left")
    .fillna({"reports_using": 0})
    .withColumn("run_timestamp", F.lit(RUN_TS).cast("timestamp")))

# ── 4. gold_report_risk : per report, risk of its model's data sources ───────
rank = (F.when(F.col("risk_tier") == "High", 3)
         .when(F.col("risk_tier") == "Medium", 2)
         .when(F.col("risk_tier") == "Low", 1).otherwise(0))
rs = (rmm.alias("r")
      .join(cmm.drop("workspace_id", "workspace_name").alias("m"),   # keep the report's workspace, not the model's
            F.col("r.dataset_id") == F.col("m.semantic_model_id"), "left")
      .withColumn("risk_rank", rank))

agg = (rs.groupBy("report_id", "report_name", "report_type",
                  "workspace_id", "workspace_name", "dataset_id")
    .agg(F.first("semantic_model_name", True).alias("semantic_model_name"),
         F.countDistinct("path").alias("n_sources"),
         F.countDistinct(F.when(F.col("risk_tier") == "High",   F.col("path"))).alias("high_sources"),
         F.countDistinct(F.when(F.col("risk_tier") == "Medium", F.col("path"))).alias("medium_sources"),
         F.countDistinct(F.when(F.col("risk_tier") == "Low",    F.col("path"))).alias("low_sources"))
    .withColumn("risk_score",
                3 * F.col("high_sources") + 2 * F.col("medium_sources") + 1 * F.col("low_sources")))

# single riskiest source per report (highest risk rank, then path)
w = Window.partitionBy("report_id").orderBy(F.col("risk_rank").desc_nulls_last(),
                                            F.col("path").asc_nulls_last())
riskiest = (rs.withColumn("rn", F.row_number().over(w)).where("rn = 1")
    .select("report_id",
            F.col("source_host").alias("riskiest_host"),
            F.col("source_kind").alias("riskiest_kind"),
            F.col("path").alias("riskiest_source"),
            F.col("risk_tier").alias("riskiest_tier")))

gold_reports = (agg.join(riskiest, "report_id", "left")
    .withColumn("run_timestamp", F.lit(RUN_TS).cast("timestamp")))

print(f"gold_file_dependencies rows: {gold_files.count()}")
print(f"gold_report_risk rows:       {gold_reports.count()}")

In [ ]:
# ── 5. Write the marts (current overwrite; Direct Lake reads these) ──────────
def write_overwrite(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(_table_uri(name))
    print(f"  {df.count():>5} rows -> {name}")

print(f"Writing to {LAKEHOUSE_NAME} (schema={LAKEHOUSE_SCHEMA}): {TABLES_PATH}")
write_overwrite(rmm,          TBL_REPORT_MAP)
write_overwrite(gold_files,   TBL_GOLD_FILES)
write_overwrite(gold_reports, TBL_GOLD_REPORTS)

## Verify — the two showcase queries

In [ ]:
spark.read.format("delta").load(_table_uri(TBL_GOLD_REPORTS)).createOrReplaceTempView("grr")
spark.read.format("delta").load(_table_uri(TBL_GOLD_FILES)).createOrReplaceTempView("gfd")

print("Top 10 riskiest reports:")
display(spark.sql(
    "SELECT report_name, workspace_name, semantic_model_name, "
    "       high_sources, medium_sources, low_sources, risk_score, "
    "       riskiest_kind, riskiest_host "
    "FROM grr ORDER BY risk_score DESC, high_sources DESC, n_sources DESC LIMIT 10"))

print("Files by source_kind (after re-classify):")
display(spark.sql(
    "SELECT source_kind, COUNT(*) AS files, SUM(models_using) AS model_links "
    "FROM gfd GROUP BY source_kind ORDER BY files DESC"))

print("Residual Unknown/Other, sliced by file extension:")
display(spark.sql(
    "SELECT COALESCE(file_extension, '(none)') AS file_extension, "
    "       COUNT(*) AS files, SUM(models_using) AS model_links "
    "FROM gfd WHERE source_kind IN ('Unknown','Other') "
    "GROUP BY COALESCE(file_extension, '(none)') ORDER BY files DESC"))

print("Files with the most dependents:")
display(spark.sql(
    "SELECT path, source_kind, source_host, risk_tier, file_extension, "
    "       models_using, reports_using, workspaces_using "
    "FROM gfd ORDER BY models_using DESC, reports_using DESC LIMIT 20"))